# Model Context Protocol

## Environment Variables & Imports

Before using this notebook, please ensure you have obtained an API Key from your LLM backend and update the `.env` file to include it as follows:

```bash
GOOGLE_API_KEY=<copy API key here>
OPENAI_API_KEY=<copy API key here>
ANTRHOPIC_API_KEY=<copy API key here>
LLAMA_CPP_API_KEY=<copy API key here>

In [ ]:
import initialize_notebook # noqa

In [ ]:
import json
import pathlib

import jinja2

from hslu.dlm03.common import backend as backend_lib
from hslu.dlm03.common import chat as chat_lib
from hslu.dlm03.common import presets
from hslu.dlm03.common.displays import ipython_display
from hslu.dlm03.tools import lint
from hslu.dlm03.util import file as file_lib

## Parameters

In [ ]:
backend_name = "Gemma 4 26B"
backend_config = presets.CONFIGS_BY_NAME[backend_name]
backend = backend_lib.LLM.from_config(backend_config)

In [ ]:
code_folder = pathlib.Path("/Users/vincent/Development/Test/")

# MCP Server

In [ ]:
from mcp.server import FastMCP

SERVER = FastMCP()


@SERVER.tool()
def list_files(path: str, glob: str) -> list[str]:
    """List the files from the given path using the given glob expression."""
    return [str(file) for file in pathlib.Path(path).glob(glob)]


@SERVER.tool()
def read_file(path: str) -> file_lib.File | str:
    """Reads the file content for the given path."""
    file_path = pathlib.Path(path)
    if not file_path.exists():
        return "File not found."
    if file_path.is_dir():
        return "Given path is a directory."
    if file_path.is_file():
        return file_lib.File.read_from(pathlib.Path(path))
    return "File could not be read."

@SERVER.tool()
def lint_file(path: str) -> lint.Issues:
    """Runs the Python linter on the given file path."""
    return lint.lint(pathlib.Path(path))

@SERVER.tool()
def write_file(path: str, content: str) -> str:
    """Runs the Python linter on the given file path."""
    file_path = pathlib.Path(path)
    try:
        file_path.write_text(content)
    except Exception as e:
        return f"Failed to write file: {e}"
    else:
        return f"Successfully wrote {file_path}"

In [ ]:
import threading

import uvicorn

PORT = 5000
HOST = "localhost"

RUN_ARGS = {
    "app": SERVER.streamable_http_app,
    "port": PORT,
    "host": HOST,
}

MCP_THREAD = threading.Thread(target=uvicorn.run, kwargs=RUN_ARGS)
MCP_THREAD.start()

# Agentic Harness

In [ ]:
import mcp
from mcp.client import streamable_http

from hslu.dlm03.common import tools as tools_lib

async with (streamable_http.streamablehttp_client(f"http://{HOST}:{PORT}/mcp") as (read_stream, write_stream, _),
            mcp.ClientSession(read_stream, write_stream) as session):
    await session.initialize()
    mcp_tools = await session.list_tools()
    tools = [tools_lib.tool_from_mcp(tool) for tool in mcp_tools.tools]

In [ ]:
system_instructions_template_string = \
"""You are an expert Software Engineer tasked to solve issues found in code.
You will be provided with some code files, and a list of issues found by a linter,
and you should output the fixed code such that the issues are not present anymore."""
system_instructions_template = jinja2.Template(system_instructions_template_string, undefined=jinja2.StrictUndefined)

In [ ]:
user_message_template_string = \
"""Please fix the files in {{ code_folder }}."""
user_message_template = jinja2.Template(user_message_template_string, undefined=jinja2.StrictUndefined)

In [ ]:
user_message = user_message_template.render(code_folder=code_folder)
system_instructions = system_instructions_template.render()

chat_display = ipython_display.IPythonChatDisplay()
chat_display.show()
chat = chat_lib.Chat(
    messages = [
        {"role": "system", "content": system_instructions},
        {"role": "user", "content": user_message},
    ],
    observers=[chat_display],
)
async with (streamable_http.streamablehttp_client(f"http://{HOST}:{PORT}/mcp") as (read_stream, write_stream, _),
            mcp.ClientSession(read_stream, write_stream) as session):
    await session.initialize()
    done = False
    while not done:
        response = backend.generate(
            chat,
            tools=tools,
        )
        done = True
        message = response.choices[0].message
        chat.append(message)
        for tool_call in message.tool_calls:
            done = False
            tool_name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)
            tool_call_result = await session.call_tool(tool_name, arguments)
            for content in tool_call_result.content:
                tool_call_result_content = tools_lib.tool_call_result_from_mcp(
                    tool_call.id,
                    content,
                )
                chat.append(tool_call_result_content)